In [1]:
import pandas as pd
df=pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [2]:
def email_assistant(email_text):
    text = str(email_text).lower().strip()

    urgent_keywords = [
        "urgent", "asap", "immediately", "submit",
        "deadline", "eod", "today", "action required"
    ]

    respond_keywords = [
        "please", "can you", "could you",
        "kindly", "review", "confirm", "reply", "respond"
    ]

    ignore_keywords = [
        "thank you", "thanks", "appreciate",
        "grateful", "well received"
    ]

    info_keywords = [
        "meeting", "schedule", "reminder",
        "update", "announcement", "notice"
    ]

    if any(keyword in text for keyword in urgent_keywords):
        return "notify", "urgent"

    elif any(keyword in text for keyword in respond_keywords):
        return "respond", "polite"

    elif any(keyword in text for keyword in ignore_keywords):
        return "ignore", "polite"

    elif any(keyword in text for keyword in info_keywords):
        return "notify", "neutral"

    else:
        return "respond", "neutral"


In [3]:
# Define dangerous actions, HITL checkpoint and human approval
dangerous_actions = ["respond"]

def hitl_check(action):
    return "Wait_for_human" if action in dangerous_actions else "Auto_approve"

def human_decision():
    decision = input("Approve action? (yes/no): ")
    return decision.lower() == "yes"

Applying HITL Check in the entire datasets

In [4]:
approved = input("Approve all WAIT_FOR_HUMAN actions? (yes/no): ").lower() == "yes"

results = []

for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    status = hitl_check(action)

    if status == "WAIT_FOR_HUMAN":
        final_action = action if approved else "blocked"
    else:
        final_action = action

    results.append({
        "email": row["body"][:50],
        "ai_action": action,
        "final_action": final_action,
        "hitl_status": status,
        "tone": tone
    })

results_df = pd.DataFrame(results)
results_df

,email,ai_action,final_action,hitl_status,tone
0,Reminder: The client meeting is scheduled at 1...,notify,notify,Auto_approve,neutral
1,Your invoice of INR 25515.09 is due on 2025-12...,respond,respond,Wait_for_human,polite
2,Reminder: The client meeting is scheduled at 1...,notify,notify,Auto_approve,neutral
3,"Hello team, please find the attached weekly re...",respond,respond,Wait_for_human,polite
4,"Hello team, please find the attached weekly re...",respond,respond,Wait_for_human,polite
...,...,...,...,...,...
195,Security alert: multiple failed login attempts...,respond,respond,Wait_for_human,neutral
196,Your order #3828 has been shipped and is expec...,respond,respond,Wait_for_human,neutral
197,Congratulations! You have been selected as a l...,respond,respond,Wait_for_human,neutral
198,Please complete the mandatory training module ...,notify,notify,Auto_approve,urgent


In [5]:
# Save results
OUT = "../data/milestone3_output_MisaKanaujiya.csv"
results_df.to_csv(OUT, index=False)
print("Saved:", OUT)


Saved: ../data/milestone3_output_MisaKanaujiya.csv
